# Aplicaciones de Redes Bayesianas y Modelo Naive Bayes

**Estudiantes:** José Vanegas y Miguel Vanegas   
**Materia:** Inteligencia Artificial  
**Tema:** Redes Bayesianas y Naive Bayes  

En esta práctica se desarrolla una aplicación sencilla de Redes Bayesianas y posteriormente se implementa un modelo de clasificación usando el algoritmo Naive Bayes.

## 1. Investigación de una Red Bayesiana

Una Red Bayesiana es un modelo probabilístico que permite representar relaciones entre variables mediante un grafo dirigido.  
Cada nodo representa una variable y cada flecha indica una relación de dependencia.

Para este trabajo se utiliza un ejemplo sencillo relacionado con el diagnóstico de diabetes.

## 1. Caso de estudio

El dataset contiene información básica de pacientes ecuatorianos.  
Las variables disponibles son:

- sexo
- ciudad
- colesterol
- edad
- diabetes

El objetivo es predecir la variable **diabetes**, usando como entrada las demás variables.

Este caso se puede representar como una Red Bayesiana porque existen variables que influyen en la probabilidad de que un paciente tenga diabetes.

## Diagrama de la Red Bayesiana



```text
                 ┌──────────┐
                 │   Sexo   │
                 └────┬─────┘
                      │
                      ▼
┌──────────┐     ┌──────────┐     ┌────────────┐
│  Ciudad  │────►│ Diabetes │◄────│ Colesterol │
└──────────┘     └────▲─────┘     └────────────┘
                      │
                      │
                 ┌────┴─────┐
                 │   Edad   │
                 └──────────┘
```
En esta red, la variable *diabetes* depende de las variables *sexo, cuidadad, colesterol y edad*

## Variables y tipos

| Variable | Tipo | Descripción |
|---|---|---|
| sexo | Categórica nominal | Representa el sexo del paciente usando valores 1 y 2 |
| ciudad | Categórica nominal | Ciudad de residencia del paciente |
| colesterol | Categórica ordinal | Nivel de colesterol: bajo, medio, alto, muy alto |
| edad | Numérica | Edad del paciente |
| diabetes | Variable objetivo | Indica si el paciente tiene diabetes: si o no |

## 2. Carga del dataset

En esta sección se carga el archivo CSV con los datos de los pacientes.  
El archivo debe llamarse **data.csv**.

In [ ]:
import pandas as pd 
import numpy as np

ruta='data.csv'
df= pd.read_csv(ruta)

df.head(10)

En esta celda se importa la librería pandas, que permite trabajar con tablas de datos.

Luego se usa `pd.read_csv("data.csv")` para cargar el archivo CSV.

Finalmente, se escribe `df` para mostrar el contenido del dataset.

In [ ]:
print("Filas y columnas del dataset:")
print(df.shape)

print("\nInformación general del dataset:")
df.info()

Filas y columnas del dataset:
(6, 5)

Información general del dataset:
<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   sexo        6 non-null      int64
 1   ciudad      6 non-null      str  
 2   colesterol  6 non-null      str  
 3   edad        6 non-null      int64
 4   diabetes    6 non-null      str  
dtypes: int64(2), str(3)
memory usage: 372.0 bytes


Esta celda permite conocer la estructura del dataset.

- `df.shape` muestra cuántas filas y columnas tiene el dataset.
- `df.info()` muestra el tipo de dato de cada columna y si existen valores nulos.

Esto ayuda a revisar si los datos están listos para ser procesados.

In [ ]:
print("Valores faltantes por columna:")
print(df.isnull().sum())

Valores faltantes por columna:
sexo          0
ciudad        0
colesterol    0
edad          0
diabetes      0
dtype: int64


Aquí se revisa si existen datos vacíos o faltantes.

El método `isnull()` identifica valores nulos y `sum()` cuenta cuántos hay por cada columna.

Si todas las columnas tienen 0 valores faltantes, significa que el dataset está completo.

## 3. Separación de variables

Para entrenar el modelo se separan los datos en:

- **X:** variables predictoras.
- **y:** variable objetivo.

En este caso:

- X = sexo, ciudad, colesterol, edad
- y = diabetes

In [ ]:
X = df[["sexo", "ciudad", "colesterol", "edad"]]
y = df["diabetes"]

print("Variables predictoras X:")
display(X)

print("Variable objetivo y:")
display(y)

Variables predictoras X:


,sexo,ciudad,colesterol,edad
0,1,Cuenca,bajo,18
1,2,Quito,alto,52
2,2,Guayaquil,medio,34
3,1,Loja,alto,61
4,2,Ambato,medio,45
5,1,Machala,muy alto,67


Variable objetivo y:


0    no
1    si
2    no
3    si
4    no
5    si
Name: diabetes, dtype: str

En esta celda se separa el dataset.

`X` contiene las variables que el modelo usará para aprender.

`y` contiene la respuesta que el modelo debe predecir, en este caso si el paciente tiene diabetes o no.

## 4. Transformación de variables

El modelo Naive Bayes necesita trabajar con valores numéricos.

Por eso se transforman las variables categóricas:

- `ciudad` se transforma con OneHotEncoder.
- `colesterol` se transforma respetando su orden: bajo < medio < alto < muy alto.
- `diabetes` se transforma en valores numéricos: no = 0, si = 1.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB

Aquí se importan las herramientas necesarias:

- `OneHotEncoder`: transforma variables nominales como ciudad.
- `OrdinalEncoder`: transforma variables ordinales como colesterol.
- `ColumnTransformer`: permite aplicar diferentes transformaciones a diferentes columnas.
- `Pipeline`: une la preparación de datos y el modelo en un solo flujo.
- `GaussianNB`: es el modelo Naive Bayes que se va a utilizar.

In [ ]:
variables_nominales = ["sexo", "ciudad"]
variables_ordinales = ["colesterol"]
variables_numericas = ["edad"]

orden_colesterol = [["bajo", "medio", "alto", "muy alto"]]

En esta celda se clasifican las variables:

- `sexo` y `ciudad` son nominales porque no tienen un orden.
- `colesterol` es ordinal porque sí tiene un orden lógico.
- `edad` es numérica.

También se define el orden correcto del colesterol:
bajo < medio < alto < muy alto.

In [ ]:
preprocesador = ColumnTransformer(
    transformers=[
        ("nominal", OneHotEncoder(handle_unknown="ignore"), variables_nominales),
        ("ordinal", OrdinalEncoder(categories=orden_colesterol), variables_ordinales),
        ("numerica", "passthrough", variables_numericas)
    ]
)

Esta celda crea el preprocesador de datos.

Se aplican tres tratamientos:

1. A las variables nominales se les aplica OneHotEncoder.
2. A la variable colesterol se le aplica OrdinalEncoder.
3. La edad se deja igual usando `passthrough`.

Esto prepara los datos para que el modelo pueda entrenar correctamente.

In [ ]:
y_transformada = y.map({"no": 0, "si": 1})

print("Variable objetivo transformada:")
print(y_transformada)

Variable objetivo transformada:
0    0
1    1
2    0
3    1
4    0
5    1
Name: diabetes, dtype: int64


Aquí se transforma la variable objetivo.

El modelo trabaja mejor con valores numéricos, por eso:

- no se convierte en 0
- si se convierte en 1

Esta variable transformada será usada para entrenar el modelo.

## 5. Modelado con Naive Bayes

En esta fase se crea y entrena el modelo Naive Bayes.

Naive Bayes es un algoritmo de clasificación basado en probabilidades.  
Se usa para predecir una clase, en este caso:

- 0 = no tiene diabetes
- 1 = sí tiene diabetes

In [ ]:
modelo = Pipeline(steps=[
    ("preprocesador", preprocesador),
    ("clasificador", GaussianNB())
])

modelo.fit(X, y_transformada)

print("Modelo Naive Bayes entrenado correctamente.")

Modelo Naive Bayes entrenado correctamente.


Aquí se construye el modelo completo.

El `Pipeline` tiene dos partes:

1. `preprocesador`: transforma las variables categóricas y ordinales.
2. `clasificador`: aplica el modelo GaussianNB.

Luego, con `modelo.fit(X, y_transformada)`, el modelo aprende a relacionar las variables de entrada con la variable diabetes.

## 6. Predicción de nuevos pacientes

Ahora se crean dos nuevos pacientes para probar el modelo.

El modelo recibirá sexo, ciudad, colesterol y edad, y devolverá una predicción:

- 0 = no tiene diabetes
- 1 = sí tiene diabetes

In [ ]:
nuevos_pacientes = pd.DataFrame({
    "sexo": [2, 1],
    "ciudad": ["Cuenca", "Loja"],
    "colesterol": ["alto", "bajo"],
    "edad": [50, 22]
})

nuevos_pacientes

,sexo,ciudad,colesterol,edad
0,2,Cuenca,alto,50
1,1,Loja,bajo,22


En esta celda se crean dos nuevos pacientes.

Paciente 1:
- sexo = 2
- ciudad = Cuenca
- colesterol = alto
- edad = 50

Paciente 2:
- sexo = 1
- ciudad = Loja
- colesterol = bajo
- edad = 22

Estos datos serán usados para probar el modelo.

In [ ]:
predicciones = modelo.predict(nuevos_pacientes)

nuevos_pacientes["prediccion_numerica"] = predicciones
nuevos_pacientes["prediccion_diabetes"] = nuevos_pacientes["prediccion_numerica"].map({
    0: "no",
    1: "si"
})

nuevos_pacientes

,sexo,ciudad,colesterol,edad,prediccion_numerica,prediccion_diabetes
0,2,Cuenca,alto,50,0,no
1,1,Loja,bajo,22,1,si


Aquí el modelo realiza la predicción.

`modelo.predict()` recibe los nuevos pacientes y devuelve una clase:

- 0 significa no tiene diabetes.
- 1 significa sí tiene diabetes.

Luego se agrega una columna con la predicción en texto para que sea más fácil de interpretar.

In [ ]:
probabilidades = modelo.predict_proba(nuevos_pacientes[["sexo", "ciudad", "colesterol", "edad"]])

tabla_probabilidades = pd.DataFrame(
    probabilidades,
    columns=["Probabilidad_no_diabetes", "Probabilidad_si_diabetes"]
)

tabla_probabilidades

,Probabilidad_no_diabetes,Probabilidad_si_diabetes
0,1.0,0.0
1,0.0,1.0


Esta celda muestra las probabilidades calculadas por el modelo.

`predict_proba()` no solo dice la clase final, sino también la probabilidad de cada clase.

Por ejemplo:

- Probabilidad de no tener diabetes.
- Probabilidad de sí tener diabetes.

Esto ayuda a justificar mejor la predicción.

In [ ]:
resultado_final = pd.concat([nuevos_pacientes, tabla_probabilidades], axis=1)

resultado_final

,sexo,ciudad,colesterol,edad,prediccion_numerica,prediccion_diabetes,Probabilidad_no_diabetes,Probabilidad_si_diabetes
0,2,Cuenca,alto,50,0,no,1.0,0.0
1,1,Loja,bajo,22,1,si,0.0,1.0


Aquí se unen los datos de los nuevos pacientes con sus predicciones y probabilidades.

Esta tabla final es útil para mostrar los resultados en el informe.

Aquí se unen los datos de los nuevos pacientes con sus predicciones y probabilidades.

Esta tabla final es útil para mostrar los resultados en el informe.

## 7. Interpretación de resultados

El primer paciente tiene 50 años y colesterol alto, por lo que el modelo puede asignarle mayor probabilidad de diabetes.

El segundo paciente tiene 22 años y colesterol bajo, por lo que el modelo puede asignarle menor probabilidad de diabetes.

Las predicciones dependen de los patrones encontrados en el dataset de entrenamiento.

## 8. Conclusiones

- Las Redes Bayesianas permiten representar relaciones entre variables mediante probabilidades.
- El caso de diabetes puede representarse como una red donde sexo, ciudad, colesterol y edad influyen en la variable diabetes.
- El modelo Naive Bayes permitió clasificar nuevos pacientes usando los datos disponibles.
- La transformación de variables categóricas fue necesaria para que el modelo pudiera trabajar correctamente.
- Aunque el dataset es pequeño, permite comprender el funcionamiento básico de un modelo probabilístico de clasificación.

## 9. Referencias APA

Scikit-learn Developers. (2026). *Naive Bayes*. Scikit-learn. https://scikit-learn.org/stable/modules/naive_bayes.html

Scikit-learn Developers. (2026). *Preprocessing data*. Scikit-learn. https://scikit-learn.org/stable/modules/preprocessing.html

Pearl, J. (1988). *Probabilistic reasoning in intelligent systems: Networks of plausible inference*. Morgan Kaufmann.